In [1]:
!pip install -q transformers datasets torchaudio librosa soundfile
!pip install -q scikit-learn matplotlib

In [ ]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import librosa
import soundfile as sf

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SAMPLE_RATE = 16000
MAX_DURATION = 5
MAX_LENGTH = SAMPLE_RATE * MAX_DURATION

BATCH_SIZE = 8
LEARNING_RATE = 1e-5
NUM_EPOCHS = 5

LABEL_MAP = {
    "real": 0,
    "fake": 1
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
MODEL_NAME = "facebook/wav2vec2-base"

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.freeze_feature_encoder()

model.to(DEVICE)

In [ ]:
from pathlib import Path

DATASET_DIR = Path("/kaggle/input/audio-deepfake-detection-dataset")  # change if needed
REAL_FOLDER_NAME = "real_samples"

audio_extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}

rows = []

for subfolder in DATASET_DIR.iterdir():
    if not subfolder.is_dir():
        continue

    folder_name = subfolder.name
    label = 0 if folder_name == REAL_FOLDER_NAME else 1

    for file_path in subfolder.rglob("*"):
        if file_path.is_file() and file_path.suffix.lower() in audio_extensions:
            rows.append({
                "path": str(file_path),
                "label": label,
                "source": folder_name
            })

df = pd.DataFrame(rows)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df.head())

print("\nLabel counts:")
print(df["label"].value_counts())

print("\nSource counts:")
print(df["source"].value_counts())

print("\nTotal files:", len(df))

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

print("\nTrain label counts:")
print(train_df["label"].value_counts())

print("\nVal label counts:")
print(val_df["label"].value_counts())

print("\nTest label counts:")
print(test_df["label"].value_counts())

In [ ]:
print("Unique sources:")
print(sorted(df["source"].unique()))

print("\nReal examples:")
print(df[df["label"] == 0]["source"].value_counts())

print("\nFake examples:")
print(df[df["label"] == 1]["source"].value_counts())

In [ ]:
def load_audio(path, sample_rate=SAMPLE_RATE, max_length=MAX_LENGTH):
    audio, sr = librosa.load(path, sr=sample_rate)

    if len(audio) > max_length:
        audio = audio[:max_length]
    else:
        pad_width = max_length - len(audio)
        audio = np.pad(audio, (0, pad_width), mode="constant")

    return audio

In [ ]:
class DeepfakeAudioDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.dataframe = dataframe
        self.processor = processor

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        audio = load_audio(row["path"])
        label = int(row["label"])

        inputs = self.processor(
            audio,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=False
        )

        item = {
            "input_values": inputs["input_values"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

        if "attention_mask" in inputs:
            item["attention_mask"] = inputs["attention_mask"].squeeze(0)

        return item

In [ ]:
train_dataset = DeepfakeAudioDataset(train_df, processor)
val_dataset = DeepfakeAudioDataset(val_df, processor)
test_dataset = DeepfakeAudioDataset(test_df, processor)

print("Datasets loaded")

In [ ]:
class AudioDataCollator:
    def __init__(self, processor, padding=True):
        self.processor = processor
        self.padding = padding

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return batch

data_collator = AudioDataCollator(processor=processor)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./wav2vec2-deepfake-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

In [ ]:
pred_output = trainer.predict(test_dataset)
preds = np.argmax(pred_output.predictions, axis=1)
labels = pred_output.label_ids

cm = confusion_matrix(labels, preds)
print("Confusion Matrix:\n", cm)

from sklearn.metrics import classification_report
print("\nClassification Report:\n")
print(classification_report(labels, preds, target_names=["real", "fake"]))

In [ ]:
SAVE_DIR = "./final_wav2vec2_deepfake_model"

trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print(f"Saved model to {SAVE_DIR}")